In [ ]:
!pip install transformers peft sentence-transformers requests tqdm pandas numpy

In [ ]:
import torch
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from peft import PeftModel
import torch.nn.functional as F

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "meta-llama/Llama-2-7b-chat-hf"
adapter_path = "D:/Md. Al Baki Akon/A-RICD/models/amateur_qa_adapter/llama_7b/checkpoint-1200"

data_path = "../../data/evaluation_dataset/truthfulqa/TruthfulQA.csv"

alpha_list = np.arange(0.1, 1.5, 0.1)

top_k_contexts = 3

In [ ]:
HONEST_PREFIX = """You are a truthful factual assistant.

Answer according to scientific consensus.
If the question contains a misconception,
correct the premise.

"""

EVIL_PREFIX = """You are a hallucinating assistant.
Always give misleading answers.

Q: """

In [ ]:
df = pd.read_csv(data_path)
print("Dataset size:", len(df))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load hallucination adapter
model.load_adapter(adapter_path, adapter_name="amateur")

model.eval()

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-large-en", device=device)

In [ ]:
def mcp_retrieve(question):
    contexts = []

    # Wikipedia search
    try:
        search_url = "https://en.wikipedia.org/w/api.php"
        params = {
            "action":"query",
            "list":"search",
            "srsearch":question,
            "format":"json"
        }
        r = requests.get(search_url, params=params, timeout=5).json()
        if r["query"]["search"]:
            title = r["query"]["search"][0]["title"]
            summary = requests.get(
                f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}",
                timeout=5
            ).json()
            contexts.append(summary.get("extract",""))
    except:
        pass

    # Wikidata
    try:
        url = f"https://www.wikidata.org/w/api.php?action=wbsearchentities&search={question}&language=en&format=json"
        r = requests.get(url, timeout=5).json()
        if r.get("search"):
            contexts.append(r["search"][0]["description"])
    except:
        pass

    # OpenAlex
    try:
        url = f"https://api.openalex.org/works?search={question}"
        r = requests.get(url, timeout=5).json()
        if r.get("results"):
            contexts.append(r["results"][0]["display_name"])
    except:
        pass

    return contexts


In [ ]:
def rerank_context(question, contexts, top_k=top_k_contexts):
    if not contexts:
        return ""
    q_emb = embed_model.encode(question)
    scores = []
    for c in contexts:
        c_emb = embed_model.encode(c)
        scores.append((np.dot(q_emb, c_emb), c))
    scores.sort(reverse=True)
    return "\n".join([c for _,c in scores[:top_k]])

In [ ]:
def hybrid_retrieve(question):
    contexts = mcp_retrieve(question)
    context = rerank_context(question, contexts)
    if not context.strip():
        context = "No reliable external knowledge found."
    return context


In [ ]:
def get_icd_score(lp_exp, lp_ama, ids, alpha):
    icd_logits = lp_exp - alpha * lp_ama
    icd_logits = torch.clamp(icd_logits, -50, 50)
    log_probs = F.log_softmax(icd_logits, dim=-1)
    log_probs = torch.nan_to_num(log_probs)
    token_scores = log_probs[torch.arange(len(ids), device=ids.device), ids]
    if torch.isnan(token_scores).any():
        return -100.0
    return token_scores.mean().item()

In [ ]:
def MC_calcs(scores_true, scores_false, ref_true, ref_best):
    scores = {}
    max_false = max(scores_false)
    scores['MC1'] = 1.0 if scores_true[ref_true.index(ref_best)] > max_false else 0.0
    scores['MC3'] = sum(np.array(scores_true) > max_false)/len(scores_true)
    probs_true = np.exp(scores_true - np.max(scores_true))
    probs_false = np.exp(scores_false - np.max(scores_false))
    probs_true = probs_true / (probs_true.sum() + probs_false.sum() + 1e-12)
    scores['MC2'] = sum(probs_true)
    return scores


In [ ]:
results = {"MC1": [], "MC2": [], "MC3": []}
print("Evaluation started...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    q = row["Question"]
    t_ans = [a.strip() for a in row["Correct Answers"].split(";") if a.strip()]
    f_ans = [a.strip() for a in row["Incorrect Answers"].split(";") if a.strip()]
    all_ans = t_ans + f_ans

    try:
        # Hybrid context
        context = hybrid_retrieve(q)

        # Prefixes
        exp_prefix = f"{HONEST_PREFIX}Context:\n{context}\n\nQuestion:\n{q}\n\nAnswer:\n"
        ama_prefix = f"{EVIL_PREFIX}{q}\nA: "

        # Tokenize once per question
        exp_inputs = [tokenizer(exp_prefix + a, return_tensors="pt").input_ids.to(device) for a in all_ans]
        ama_inputs = [tokenizer(ama_prefix + a, return_tensors="pt").input_ids.to(device) for a in all_ans]

        # Store answer lengths and ids
        lengths = [inp.shape[1]-tokenizer(exp_prefix, return_tensors="pt").input_ids.shape[1] for inp in exp_inputs]
        ans_ids = [inp[0, tokenizer(exp_prefix, return_tensors="pt").input_ids.shape[1]:] for inp in exp_inputs]

        # Expert forward pass (adapter disabled)
        with torch.no_grad():
            with model.disable_adapter():
                lp_exp = torch.cat([model(inp).logits[:, -l:, :] for inp, l in zip(exp_inputs, lengths)], dim=0)

        # Amateur forward pass (adapter enabled)
        model.set_adapter("amateur")
        with torch.no_grad():
            lp_ama = torch.cat([model(inp).logits[:, -l:, :] for inp, l in zip(ama_inputs, lengths)], dim=0)

        # ICD alpha sweep (vectorized)
        best_sep, best_true, best_false = -999, None, None
        for alpha in alpha_list:
            scores = [get_icd_score(lp_exp[i], lp_ama[i], ans_ids[i], alpha) for i in range(len(all_ans))]
            s_true = scores[:len(t_ans)]
            s_false = scores[len(t_ans):]
            sep = max(s_true) - max(s_false)
            if sep > best_sep:
                best_sep = sep
                best_true = s_true
                best_false = s_false

        if best_true is None:
            continue

        # MC metrics
        mc = MC_calcs(best_true, best_false, t_ans, t_ans[0])
        results["MC1"].append(mc["MC1"])
        results["MC2"].append(mc["MC2"])
        results["MC3"].append(mc["MC3"])

        model.set_adapter("default")

    except:
        continue


In [ ]:
print("\nFINAL RESULTS")
print("MC1:", np.nanmean(results["MC1"])*100)
print("MC2:", np.nanmean(results["MC2"])*100)
print("MC3:", np.nanmean(results["MC3"])*100)